# Cartpole
## Observation Space

In CartPole-v1, the observation is a 4-dimensional continuous array representing the state of the system:

1. **Cart Position** (x): Horizontal position of the cart on the track (range: -2.4 to 2.4)
2. **Cart Velocity** (ẋ): Speed of the cart moving left or right (range: -∞ to ∞)
3. **Pole Angle** (θ): Angle of the pole from vertical in radians (range: -0.209 to 0.209 radians, which is approximately ±12°)
4. **Pole Angular Velocity** (θ̇): Rate of change of the pole angle (range: -∞ to ∞)

### Current Observation Example
Based on your current state: `[0.086, 0.995, -0.093, -1.551]`
- Cart is slightly to the right (+0.086)
- Cart is moving to the right (+0.995)
- Pole is tilted slightly to the left (-0.093 radians)
- Pole is rotating to the left (-1.551 rad/s)

The goal is to keep the pole balanced upright (angle near 0) and the cart centered (position near 0) for as long as possible.

In [200]:
def observation_printer(observation):
    cart_position, cart_velocity, pole_angle, pole_angular_velocity = observation
    print(f"Cart Position: {cart_position:.4f} m")
    print(f"Cart Velocity: {cart_velocity:.4f} m/s")
    print(f"Pole Angle: {np.degrees(pole_angle):.2f}°")
    print(f"Pole Angular Velocity: {np.degrees(pole_angular_velocity):.2f}°/s")

In [201]:
# lean direction policy
def policy_lean_direction(observation):
    angle = observation[2]
    location = observation[0]
    if angle < 0:
        return 0  # Move left
    else:
        return 1  # Move right

In [202]:
import gymnasium as gym
import numpy as np

def run_experiment(weights, num_episodes=50, max_steps=5000, render_mode=None, verbose=False):
    """
    Runs CartPole-v1 for num_episodes using the provided weights for the policy.
    weights: [angle, angular_velocity, location, velocity]
    Returns average reward over episodes.
    """
    env = gym.make('CartPole-v1', render_mode=render_mode)
    
    def policy_weighted_custom(observation):
        angle = observation[2]
        angular_velocity = observation[3]
        location = observation[0]
        velocity = observation[1]
        score = (weights[0] * angle) + (weights[1] * angular_velocity) + (weights[2] * location) + (weights[3] * velocity)
        return 0 if score < 0 else 1

    total_reward = 0
    min_reward = float('inf')
    max_reward = 0

    for i in range(num_episodes):
        observation, info = env.reset()
        episode_reward = 0
        for step in range(max_steps):
            action = policy_weighted_custom(observation)
            observation, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            if terminated or truncated:
                break
        total_reward += episode_reward
        min_reward = min(min_reward, episode_reward)
        max_reward = max(max_reward, episode_reward)
        if verbose:
            print(f"Episode {i+1}: reward={episode_reward}")
    env.close()
    avg_reward = total_reward / num_episodes
    return avg_reward, min_reward, max_reward

In [205]:
# Example usage:
weights = [0, 0, 0, 0]  # [angle, angular_velocity, location, velocity]
avg, min_r, max_r = run_experiment(weights, num_episodes=10, render_mode="human" , verbose=True)
print(f"Average reward: {avg:.2f}, Min: {min_r}, Max: {max_r}")

Episode 1: reward=10.0
Episode 2: reward=9.0
Episode 3: reward=8.0
Episode 4: reward=10.0
Episode 5: reward=10.0
Episode 6: reward=9.0
Episode 7: reward=8.0
Episode 8: reward=8.0
Episode 9: reward=9.0
Episode 10: reward=8.0
Average reward: 8.90, Min: 8.0, Max: 10.0


In [206]:
def optimize_weights(initial_weights, perturbation=0.2, num_episodes=200, max_steps=5000, max_iters=20, verbose=True):
    """
    Simple coordinate-wise optimization for policy weights, maximizing the minimum score across episodes.
    """
    weights = np.array(initial_weights, dtype=float)
    direction = np.ones_like(weights)
    _, min_score, _ = run_experiment(weights, num_episodes, max_steps)
    if verbose:
        print(f"Initial weights: {weights}, baseline min score: {min_score:.2f}")
    
    for iteration in range(max_iters):
        improved = False
        for i in range(len(weights)):
            test_weights = weights.copy()
            test_weights[i] += perturbation * direction[i]
            _, test_min_score, _ = run_experiment(test_weights, num_episodes, max_steps)
            if verbose:
                print(f"  Perturb param {i}: {test_weights} -> min score {test_min_score:.2f}")
            if test_min_score > min_score:
                weights[i] += perturbation * direction[i]
                min_score = test_min_score
                improved = True
            else:
                direction[i] *= -1  # Flip direction for next time
        if not improved:
            if verbose: print(f"No improvement in iteration {iteration}, stopping.")
            break
        if verbose: print(f"Iteration {iteration}: weights {weights}, min_score {min_score:.2f}")
    return weights, min_score

In [207]:
weights, score = optimize_weights([1.0, 0.5, 0.5, 0.1], verbose=True)

Initial weights: [1.  0.5 0.5 0.1], baseline min score: 120.00
  Perturb param 0: [1.2 0.5 0.5 0.1] -> min score 190.00
  Perturb param 1: [1.2 0.7 0.5 0.1] -> min score 166.00
  Perturb param 2: [1.2 0.5 0.7 0.1] -> min score 103.00
  Perturb param 3: [1.2 0.5 0.5 0.3] -> min score 500.00
Iteration 0: weights [1.2 0.5 0.5 0.3], min_score 500.00
  Perturb param 0: [1.4 0.5 0.5 0.3] -> min score 500.00
  Perturb param 1: [1.2 0.3 0.5 0.3] -> min score 500.00
  Perturb param 2: [1.2 0.5 0.3 0.3] -> min score 500.00
  Perturb param 3: [1.2 0.5 0.5 0.5] -> min score 500.00
No improvement in iteration 1, stopping.


In [208]:
print("Weights:", weights)
print("Min Score:", score)

Weights: [1.2 0.5 0.5 0.3]
Min Score: 500.0


In [209]:
# Visualize the best learned weights using the human viewer
def visualize_solution(weights, max_steps=5000):
    import gymnasium as gym
    import time
    env = gym.make('CartPole-v1', render_mode='human')
    observation, info = env.reset()
    def policy_weighted_custom(observation):
        angle = observation[2]
        angular_velocity = observation[3]
        location = observation[0]
        velocity = observation[1]
        score = (weights[0] * angle) + (weights[1] * angular_velocity) + (weights[2] * location) + (weights[3] * velocity)
        return 0 if score < 0 else 1
    total_reward = 0
    for step in range(max_steps):
        action = policy_weighted_custom(observation)
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break
    env.close()
    print(f"Total reward in visualization: {total_reward}")
# Example: visualize_solution([1.0, 0.5, 0.5, 0.1])

In [211]:
visualize_solution(weights)

Total reward in visualization: 500.0
